In [5]:
import os
import requests
import mimetypes
from duckduckgo_search import DDGS

def get_food_array(file_path):
    """ขั้นที่ 1: อ่านไฟล์และสร้าง Array ข้อมูลพร้อม ID"""
    food_array = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            # สร้าง List ของ Dictionary โดยใช้ Index เป็น ID
            lines = [line.strip() for line in f if line.strip()]
            for index, name in enumerate(lines):
                food_array.append({
                    "id": index,
                    "name": name
                })
        return food_array
    except FileNotFoundError:
        print(f"❌ ไม่พบไฟล์ {file_path}")
        return []

def download_images(food_array):
    """ขั้นที่ 2: ดาวน์โหลดรูป โดยจะข้ามถ้ามีไฟล์อยู่แล้ว"""
    folder_name = "food_images"
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
    }

    with DDGS() as ddgs:
        for item in food_array:
            food_id = item['id']
            food_name = item['name']
            
            # --- ตรวจสอบว่ามีไฟล์ ID นี้อยู่ในโฟลเดอร์หรือยัง ---
            # เราจะเช็คไฟล์ทุกนามสกุลที่อาจจะเป็นไปได้ (jpg, png, webp, jpeg)
            existing_files = [f for f in os.listdir(folder_name) if f.startswith(f"{food_id}.")]
            
            if existing_files:
                print(f"⏩ ข้าม [{food_id}]: {food_name} (มีไฟล์อยู่แล้ว: {existing_files[0]})")
                continue

            print(f"🔍 กำลังค้นหา [{food_id}]: {food_name}...")
            
            try:
                results = ddgs.images(keywords=food_name, region="wt-wt", max_results=1)
                if not results:
                    print(f"❓ ไม่พบรูป: {food_name}")
                    continue

                image_url = results[0]['image']
                response = requests.get(image_url, headers=headers, timeout=15)
                response.raise_for_status()

                # ตรวจสอบนามสกุล
                content_type = response.headers.get('content-type')
                extension = mimetypes.guess_extension(content_type)
                if not extension or extension == '.jpe': extension = '.jpg'

                filename = f"{food_id}{extension}"
                file_path = os.path.join(folder_name, filename)
                
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                
                print(f"✅ บันทึกแล้ว: {filename}")

            except Exception as e:
                print(f"❌ พลาด ({food_name}): {e}")

# --- ส่วนการรันโปรแกรม ---

file_name = "data.txt"

# 1. สร้าง Array ออกมาก่อน
my_food_list = get_food_array(file_name)

# 2. แสดงข้อมูลที่เตรียมไว้ให้ผู้ใช้ตรวจสอบ
print("--- ตรวจสอบข้อมูลที่เตรียมไว้ ---")
for item in my_food_list:
    print(f"ID: {item['id']} | ชื่ออาหาร: {item['name']}")
print(f"รวมทั้งหมด {len(my_food_list)} รายการ")
print("------------------------------\n")

# 3. เริ่มโหลดรูป
if my_food_list:
    confirm = input("กด Enter เพื่อเริ่มดาวน์โหลด (หรือพิมพ์ 'n' เพื่อยกเลิก): ")
    if confirm.lower() != 'n':
        download_images(my_food_list)
    else:
        print("ยกเลิกการทำงาน")

--- ตรวจสอบข้อมูลที่เตรียมไว้ ---
ID: 0 | ชื่ออาหาร: ผัดกะเพราหมูสับ/หมูชิ้น
ID: 1 | ชื่ออาหาร: ผัดกะเพราไก่
ID: 2 | ชื่ออาหาร: ผัดกะเพราเนื้อ
ID: 3 | ชื่ออาหาร: ผัดกะเพรากุ้ง
ID: 4 | ชื่ออาหาร: ผัดกะเพราปลาหมึก
ID: 5 | ชื่ออาหาร: ผัดกะเพราทะเล
ID: 6 | ชื่ออาหาร: ผัดกะเพราปลา
ID: 7 | ชื่ออาหาร: ผัดกะเพราหมูยอ
ID: 8 | ชื่ออาหาร: ผัดกะเพราปลาดุก
ID: 9 | ชื่ออาหาร: ผัดกะเพราหมูกรอบ
ID: 10 | ชื่ออาหาร: ผัดกะเพราเครื่องใน
ID: 11 | ชื่ออาหาร: ผัดกะเพราไข่เยี่ยวม้า
ID: 12 | ชื่ออาหาร: ผัดกะเพราเต้าหู้หมูสับ
ID: 13 | ชื่ออาหาร: ผัดกะเพราวุ้นเส้นหมูสับ
ID: 14 | ชื่ออาหาร: ผัดกะเพราไข่เจียว/ไข่ดาว
ID: 15 | ชื่ออาหาร: ผัดพริกแกงไก่
ID: 16 | ชื่ออาหาร: ผัดพริกแกงหมูชิ้น
ID: 17 | ชื่ออาหาร: ผัดพริกแกงหมูกรอบ
ID: 18 | ชื่ออาหาร: ผัดพริกแกงปลาดุก
ID: 19 | ชื่ออาหาร: ผัดพริกแกงเครื่องใน
ID: 20 | ชื่ออาหาร: ผัดพริกแกงกุ้ง
ID: 21 | ชื่ออาหาร: ผัดพริกแกงทะเล
ID: 22 | ชื่ออาหาร: ผัดพริกแกงผักบุ้ง & หมูกรอบ
ID: 23 | ชื่ออาหาร: ผัดพริกแกงผักบุ้ง & ปลาดุก
ID: 24 | ชื่ออาหาร: ผัดผงกะหรี่กุ้ง
ID: 25 | ชื่ออาหา

/var/folders/s8/ndjgp9ms19v0htmk4jvg9v5m0000gn/T/ipykernel_74672/3182274589.py:33: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔍 กำลังค้นหา [0]: ผัดกะเพราหมูสับ/หมูชิ้น...
✅ บันทึกแล้ว: 0.jpg
🔍 กำลังค้นหา [1]: ผัดกะเพราไก่...
✅ บันทึกแล้ว: 1.jpg
🔍 กำลังค้นหา [2]: ผัดกะเพราเนื้อ...
✅ บันทึกแล้ว: 2.jpg
🔍 กำลังค้นหา [3]: ผัดกะเพรากุ้ง...
✅ บันทึกแล้ว: 3.jpg
🔍 กำลังค้นหา [4]: ผัดกะเพราปลาหมึก...
✅ บันทึกแล้ว: 4.jpg
🔍 กำลังค้นหา [5]: ผัดกะเพราทะเล...
✅ บันทึกแล้ว: 5.jpg
🔍 กำลังค้นหา [6]: ผัดกะเพราปลา...
✅ บันทึกแล้ว: 6.jpg
🔍 กำลังค้นหา [7]: ผัดกะเพราหมูยอ...
✅ บันทึกแล้ว: 7.jpg
🔍 กำลังค้นหา [8]: ผัดกะเพราปลาดุก...
✅ บันทึกแล้ว: 8.jpg
🔍 กำลังค้นหา [9]: ผัดกะเพราหมูกรอบ...
✅ บันทึกแล้ว: 9.jpg
🔍 กำลังค้นหา [10]: ผัดกะเพราเครื่องใน...
✅ บันทึกแล้ว: 10.jpg
🔍 กำลังค้นหา [11]: ผัดกะเพราไข่เยี่ยวม้า...
✅ บันทึกแล้ว: 11.jpg
🔍 กำลังค้นหา [12]: ผัดกะเพราเต้าหู้หมูสับ...
✅ บันทึกแล้ว: 12.jpg
🔍 กำลังค้นหา [13]: ผัดกะเพราวุ้นเส้นหมูสับ...
✅ บันทึกแล้ว: 13.jpg
🔍 กำลังค้นหา [14]: ผัดกะเพราไข่เจียว/ไข่ดาว...
✅ บันทึกแล้ว: 14.jpg
🔍 กำลังค้นหา [15]: ผัดพริกแกงไก่...
✅ บันทึกแล้ว: 15.jpg
🔍 กำลังค้นหา [16]: ผัดพริกแกงหมูชิ้น...
✅ บันทึ

# Upload

In [41]:
for index,i in enumerate(my_food_list):

    prices = 0
    if "เนื้อ" in i['name']:
        prices = 50
    elif "หมู" in i['name'] or "ไก่" in i['name']:
        prices = 40
    elif "ปลา" in i['name']:
        prices = 45
    elif "ต้ม" in i['name']:
        prices = 60
    else:
        prices = 50
    my_food_list[index]['price'] = prices
my_food_list

[{'id': 29, 'name': 'ผัดผงกะหรี่ปลาหมึก', 'price': 45},
 {'id': 30, 'name': 'ผัดผงกะหรี่ทะเล', 'price': 50},
 {'id': 31, 'name': 'หมูทอดกระเทียม', 'price': 40},
 {'id': 32, 'name': 'ไก่ทอดกระเทียม', 'price': 40},
 {'id': 33, 'name': 'กุ้งกระเทียมพริกไทย', 'price': 50},
 {'id': 34, 'name': 'ปลาหมึกกระเทียมพริกไทย', 'price': 45},
 {'id': 35, 'name': 'หมูกรอบกระเทียมพริกไทย (เด็ด!)', 'price': 40},
 {'id': 36, 'name': 'หมูสับกระเทียมพริกไทย', 'price': 40},
 {'id': 37, 'name': 'เนื้อกระเทียมพริกไทย', 'price': 50},
 {'id': 38, 'name': 'เต้าหู้ไข่ & หมูสับกระเทียมพริกไทย', 'price': 40},
 {'id': 39, 'name': 'ตับหมูกระเทียมพริกไทย', 'price': 40},
 {'id': 40, 'name': 'หมูผัดน้ำมันหอย', 'price': 40},
 {'id': 41, 'name': 'ไก่ผัดน้ำมันหอย', 'price': 40},
 {'id': 42, 'name': 'เนื้อผัดน้ำมันหอย', 'price': 50},
 {'id': 43, 'name': 'กุ้งผัดน้ำมันหอย', 'price': 50},
 {'id': 44, 'name': 'หมูกรอบผัดน้ำมันหอย', 'price': 40},
 {'id': 45, 'name': 'ตับผัดน้ำมันหอย', 'price': 50},
 {'id': 46, 'name': 'ทะเลผัดฉ

In [42]:
my_food_list

[{'id': 29, 'name': 'ผัดผงกะหรี่ปลาหมึก', 'price': 45},
 {'id': 30, 'name': 'ผัดผงกะหรี่ทะเล', 'price': 50},
 {'id': 31, 'name': 'หมูทอดกระเทียม', 'price': 40},
 {'id': 32, 'name': 'ไก่ทอดกระเทียม', 'price': 40},
 {'id': 33, 'name': 'กุ้งกระเทียมพริกไทย', 'price': 50},
 {'id': 34, 'name': 'ปลาหมึกกระเทียมพริกไทย', 'price': 45},
 {'id': 35, 'name': 'หมูกรอบกระเทียมพริกไทย (เด็ด!)', 'price': 40},
 {'id': 36, 'name': 'หมูสับกระเทียมพริกไทย', 'price': 40},
 {'id': 37, 'name': 'เนื้อกระเทียมพริกไทย', 'price': 50},
 {'id': 38, 'name': 'เต้าหู้ไข่ & หมูสับกระเทียมพริกไทย', 'price': 40},
 {'id': 39, 'name': 'ตับหมูกระเทียมพริกไทย', 'price': 40},
 {'id': 40, 'name': 'หมูผัดน้ำมันหอย', 'price': 40},
 {'id': 41, 'name': 'ไก่ผัดน้ำมันหอย', 'price': 40},
 {'id': 42, 'name': 'เนื้อผัดน้ำมันหอย', 'price': 50},
 {'id': 43, 'name': 'กุ้งผัดน้ำมันหอย', 'price': 50},
 {'id': 44, 'name': 'หมูกรอบผัดน้ำมันหอย', 'price': 40},
 {'id': 45, 'name': 'ตับผัดน้ำมันหอย', 'price': 50},
 {'id': 46, 'name': 'ทะเลผัดฉ

In [40]:
import requests

# 1. กำหนด URL ของ Next.js API
url = "http://localhost:3000/api/item" # เปลี่ยนเป็น URL จริงของคุณ
for i in my_food_list:
    try:
        # 2. เตรียมข้อมูล (ที่เป็น String/Text)
        data = {
            "Name": i['name'],
            "shopId": "58d24f11-8a4f-47c9-bad8-46b30bf8b658",
            "price": i['price'],
        }

        # 3. เตรียมไฟล์ภาพ
        # 'rb' หมายถึง read binary (จำเป็นสำหรับการส่งไฟล์)
        files = {
            "image": (f"{i['id']}.jpg", open(f"food_images/{i['id']}.jpg", "rb"), "image/jpeg")
        }

        # 4. ส่ง Request ไปยัง API
        try:
            response = requests.post(url, data=data, files=files)
            
            # ตรวจสอบผลลัพธ์
            if response.status_code == 200:
                print("ส่งข้อมูลสำเร็จ!")
                print(response.json())
            else:
                print(f"เกิดข้อผิดพลาด: {response.status_code}")
                print(response.text)

        except Exception as e:
            print(f"Connection Error: {e}")

        finally:
            # อย่าลืมปิดไฟล์เพื่อคืน Memory
            files["image"][1].close()
    except Exception as e:
        print(f"Error with item {i['name']}: {e}")

Error with item ผัดผงกะหรี่ปลาหมึก: [Errno 2] No such file or directory: 'food_images/29.jpg'
ส่งข้อมูลสำเร็จ!
{'success': True, 'id': '0aa5e9ce-a1c9-414b-9be6-0eb58fc27083'}
ส่งข้อมูลสำเร็จ!
{'success': True, 'id': '13bb3559-740b-4a6c-aa8c-24a34ba7fe21'}
ส่งข้อมูลสำเร็จ!
{'success': True, 'id': '314c8d1c-5e33-4ffe-8daf-acfef291fa5b'}
Error with item กุ้งกระเทียมพริกไทย: [Errno 2] No such file or directory: 'food_images/33.jpg'
ส่งข้อมูลสำเร็จ!
{'success': True, 'id': '22217cf0-c5a6-4bf3-9b32-0ca486839b0b'}
Error with item หมูกรอบกระเทียมพริกไทย (เด็ด!): [Errno 2] No such file or directory: 'food_images/35.jpg'
ส่งข้อมูลสำเร็จ!
{'success': True, 'id': '87d17f35-be40-45db-ad64-ba4fbee46123'}
ส่งข้อมูลสำเร็จ!
{'success': True, 'id': '161738f2-eb9c-4a02-8636-f44721a8c24b'}
ส่งข้อมูลสำเร็จ!
{'success': True, 'id': '847be6c8-3bef-4b4b-b351-b1d6473c51e1'}
ส่งข้อมูลสำเร็จ!
{'success': True, 'id': '6f92312e-fb05-439b-80b1-1d21ae9ba23f'}
ส่งข้อมูลสำเร็จ!
{'success': True, 'id': '542d642b-16b4-4